# Module 04 — Control Flow, Functions, and Scope

## Exercise 04.5 — Six loops to rewrite

Each one works. Each one is written the way someone writes Python when they are
still writing C or Java in it. Rewrite each idiomatically, keeping the
behaviour identical, and note in a comment what the rewrite eliminated.
Run:  python ex05_control_flow.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Loops, and the `else` nobody expects

In [ ]:
for item in collection:      # iterates ANYTHING iterable (Module 14)
    ...

for i, item in enumerate(collection, start=1):
    ...

for a, b in zip(xs, ys, strict=True):    # strict=True is 3.10+ and you want it
    ...

for key, value in mapping.items():
    ...

`zip(strict=True)` raises if the iterables have different lengths. Without it,
`zip` silently stops at the shortest, which has hidden many data bugs. Default
to `strict=True` unless truncation is genuinely intended.

### `for ... else`

The `else` clause runs **if the loop completed without `break`**. It is not "if
the loop body never ran".

In [ ]:
for user in users:
    if user.is_admin:
        print("found an admin")
        break
else:
    print("no admin found")        # runs only if we never broke out

Read `else` here as `nobreak` and it becomes obvious. It exists to remove the
`found = False` flag variable:

In [ ]:
found = False                       # the pattern for ... else replaces
for user in users:
    if user.is_admin:
        found = True
        break
if not found:
    ...

It is rare in real code, and it is on every Python quiz.

### Loop control

```text
break        # exit the innermost loop
continue     # next iteration
```


There is no labelled break. To exit nested loops, either extract the loops into
a function and `return`, or use a flag, or iterate a product:

In [ ]:
from itertools import product
for i, j in product(range(n), range(m)):
    if done(i, j):
        break                       # one loop, so one break is enough

Extracting to a function is almost always the cleanest of the three.

### Do not mutate what you are iterating

In [ ]:
for x in items:
    if pred(x):
        items.remove(x)             # silently skips elements (Module 02, q11)

items = [x for x in items if not pred(x)]      # correct
items[:] = [x for x in items if not pred(x)]   # correct, and in place

---

## Concept 3. `match`: structural pattern matching, not a switch

`match` (3.10+) destructures values. Using it as a C-style switch wastes it.

In [ ]:
match command.split():
    case ["go", direction]:
        move(direction)
    case ["take", *items]:                 # capture the rest
        for item in items:
            take(item)
    case ["quit" | "exit"]:                # alternatives
        raise SystemExit
    case []:
        print("say something")
    case _:                                 # the default; _ matches anything
        print(f"unknown: {command}")

It matches structure, types, and attributes:

In [ ]:
match event:
    case {"type": "click", "pos": (x, y)}:          # dict + tuple shape
        handle_click(x, y)
    case {"type": "key", "code": int() as code}:    # type check + capture
        handle_key(code)
    case Point(x=0, y=0):                            # class patterns
        print("origin")
    case Point(x=x, y=y) if x == y:                  # a guard
        print("diagonal")

Two traps:

**A bare name is a capture, not a comparison.**

```text
case OK:              # binds anything to the name OK. Always matches!
case Status.OK:       # a dotted name IS compared. This is what you meant.
```


This is the number one `match` bug. Any pattern that is a plain identifier
captures; only dotted names, literals, and class patterns compare.

**Class patterns need `__match_args__`** for positional matching, which
`@dataclass` provides automatically (Module 11).

When is `match` worth it? When you are destructuring nested data — parsing,
protocol handling, AST walking, event dispatch. For dispatching on a single
value, a dict of functions is clearer and faster.

---

## Concept 5. Scope: LEGB

Name lookup walks four scopes, in order:

```text
L  Local        the current function's own names
E  Enclosing    any enclosing function's names (closures)
G  Global       the module's top-level names
B  Builtins     print, len, list, ...
```


In [ ]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print(x)        # local
    inner()
    print(x)            # enclosing
outer()
print(x)                # global

### Assignment makes a name local for the whole function

This is the rule that produces the most confusing error in the language:

In [ ]:
counter = 0

def increment():
    counter += 1        # UnboundLocalError: local variable 'counter'
                        # referenced before assignment

The compiler scans the function body *before* it runs. It sees `counter` being
assigned somewhere in the body, so `counter` is a **local** for the entire
function — including on the line that reads it, which happens before any write.
Reading it there is reading an unassigned local.

Note the asymmetry that makes this so confusing:

In [ ]:
def read_only():
    print(counter)      # fine -- no assignment in this body, so it is global

def mutate_ok():
    items.append(1)     # fine -- MUTATION is not assignment

The fixes, in order of preference:

In [ ]:
def increment(counter: int) -> int:      # 1. best: take it in, hand it back
    return counter + 1

class Counter:                            # 2. state belongs in an object
    def __init__(self): self.n = 0
    def increment(self): self.n += 1

def increment():                          # 3. last resort
    global counter
    counter += 1

`global` is almost always a design smell. It makes a function's behaviour depend
on invisible state and makes it untestable in isolation.

### `nonlocal` for closures

In [ ]:
def make_counter():
    count = 0
    def increment():
        nonlocal count       # rebind the ENCLOSING count, not a new local
        count += 1
        return count
    return increment

c = make_counter()
c(); c(); c()          # 1, 2, 3

`global` reaches the module scope. `nonlocal` reaches the nearest enclosing
*function* scope. Neither reaches a class body.

### Comprehensions have their own scope

In [ ]:
i = "untouched"
squares = [i * i for i in range(5)]
print(i)                # 'untouched' -- the loop variable did not leak

True since Python 3. A plain `for` loop *does* leak its variable; a comprehension
does not.

---

## Concept 7. Type hints

Hints are not enforced at runtime. They are checked by mypy or pyright, read by
your editor, and used by libraries like Pydantic and FastAPI. Module 17 is the
full treatment; this is the working subset.

In [ ]:
def greet(name: str, times: int = 1) -> str: ...

def parse(raw: str) -> dict[str, int]: ...            # builtin generics, 3.9+
def find(xs: list[int]) -> int | None: ...            # union syntax, 3.10+
def apply(fn: Callable[[int], str], x: int) -> str: ...

from collections.abc import Iterable, Sequence
def total(values: Iterable[float]) -> float: ...      # accept ANY iterable

Two habits worth forming now:

**Accept the widest type, return the narrowest.** Take `Iterable[str]`, not
`list[str]` — then a generator, a tuple, or a set all work. Return `list[str]`,
not `Iterable[str]` — then the caller knows they can index it.

**`X | None` is not optional-as-in-omittable**; it means the value may be `None`.
A parameter is omittable because it has a default.

Write hints on every function you write in this course. Not because Python needs
them, but because writing the return type forces you to decide what the function
actually produces — which is where half of all design bugs are found.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Statements and expressions
- Section 2: Loops, and the `else` nobody expects
- Section 3: `match`: structural pattern matching, not a switch
- Section 4: Functions: the six kinds of parameter
- Section 5: Scope: LEGB
- Section 6: Closures
- Section 7: Type hints

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Iterable


# --- 1: index arithmetic ------------------------------------------------------

---

## `with_positions`

ELIMINATE: manual index tracking.

In [ ]:
def with_positions(names: list[str]) -> list[str]:
    """ELIMINATE: manual index tracking."""
    result = []
    i = 0
    while i < len(names):
        result.append(f"{i + 1}. {names[i]}")
        i += 1
    return result

---

## `pair_up`

ELIMINATE: indexing two lists in lockstep.

In [ ]:
def pair_up(names: list[str], scores: list[int]) -> list[str]:
    """ELIMINATE: indexing two lists in lockstep.
    BONUS: make mismatched lengths an ERROR rather than silent truncation."""
    result = []
    for i in range(len(names)):
        result.append(f"{names[i]}={scores[i]}")
    return result

---

## `has_admin`

ELIMINATE: the found flag. Two idiomatic rewrites exist -- one uses

In [ ]:
def has_admin(users: list[dict[str, object]]) -> bool:
    """ELIMINATE: the found flag. Two idiomatic rewrites exist -- one uses
    for/else, the other uses a builtin. Write both, and say which you prefer."""
    found = False
    for user in users:
        if user.get("role") == "admin":
            found = True
            break
    return found

---

## `split_valid`

ELIMINATE: two appends in a manual loop.

In [ ]:
def split_valid(records: list[dict[str, object]]) -> tuple[list, list]:  # type: ignore[type-arg]
    """ELIMINATE: two appends in a manual loop.
    Careful: a single comprehension iterates twice. Is that acceptable here?
    When is it not?"""
    valid = []
    invalid = []
    for record in records:
        if record.get("email"):
            valid.append(record)
        else:
            invalid.append(record)
    return valid, invalid

---

## `find_pair`

ELIMINATE: the nested break dance.

In [ ]:
def find_pair(numbers: list[int], target: int) -> tuple[int, int] | None:
    """ELIMINATE: the nested break dance.
    Two rewrites: one using itertools, one using a dict for an O(n) algorithm.
    The second is a genuinely better algorithm, not just tidier code -- say what
    changed in the complexity."""
    result = None
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] + numbers[j] == target:
                result = (numbers[i], numbers[j])
                break
        if result is not None:
            break
    return result

---

## `render_csv`

ELIMINATE: string concatenation in a loop, and the trailing-separator

In [ ]:
def render_csv(rows: list[list[str]]) -> str:
    """ELIMINATE: string concatenation in a loop, and the trailing-separator
    dance."""
    out = ""
    for row in rows:
        line = ""
        for i, cell in enumerate(row):
            line += cell
            if i < len(row) - 1:
                line += ","
        out += line + "\n"
    return out

---

## `test_all`

_test all_

In [ ]:
def test_all() -> None:
    assert with_positions(["a", "b"]) == ["1. a", "2. b"]
    assert pair_up(["a", "b"], [1, 2]) == ["a=1", "b=2"]
    assert has_admin([{"role": "user"}, {"role": "admin"}]) is True
    assert has_admin([{"role": "user"}]) is False
    valid, invalid = split_valid([{"email": "a@b"}, {}, {"email": ""}])
    assert len(valid) == 1 and len(invalid) == 2
    assert find_pair([1, 2, 3, 4], 7) == (3, 4)
    assert find_pair([1, 2], 99) is None
    assert render_csv([["a", "b"], ["c", "d"]]) == "a,b\nc,d\n"
    print("  PASS  behaviour preserved")

---

## `test_pair_up_is_strict`

_test pair up is strict_

In [ ]:
def test_pair_up_is_strict() -> None:
    try:
        pair_up(["a", "b", "c"], [1, 2])
    except ValueError:
        print("  PASS  pair_up rejects mismatched lengths")
        return
    print("  TODO  pair_up still truncates silently")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    test_all()
    test_pair_up_is_strict()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.